In [ ]:
from google.colab import drive
drive.mount('/content/drive')

checkpoint = torch.load('/content/drive/MyDrive/diabetes_model.pth', weights_only=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from google.colab import drive

# ── Mount Drive & load model ─────────────────────────────────────────
drive.mount('/content/drive')

checkpoint = torch.load('/content/drive/MyDrive/diabetes_model.pth', weights_only=False)
X_mean = checkpoint['X_mean']
X_std  = checkpoint['X_std']

# ── Rebuild exact same model architecture ────────────────────────────
class DiabetesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1),  nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = DiabetesNet()
model.load_state_dict(checkpoint['model_state'])
model.eval()
print("Model loaded!")

# ── Upload diabetes.csv from Kaggle ──────────────────────────────────
from google.colab import files
uploaded = files.upload()   # select diabetes.csv when prompted

# ── Load & fix real data ─────────────────────────────────────────────
df_real = pd.read_csv('diabetes.csv')

zero_cols = ['Glucose', 'BloodPressure', 'Insulin', 'BMI']
df_real[zero_cols] = df_real[zero_cols].replace(0, np.nan)
for col in zero_cols:
    df_real[col] = df_real[col].fillna(df_real[col].median())

# ── Pick the 5 columns the model was trained on ──────────────────────
X_real = df_real[['Age', 'BMI', 'Glucose',
                   'BloodPressure', 'Insulin']].values.astype(np.float32)
y_real = df_real['Outcome'].values

# ── Normalize using TRAINING stats (not real data stats) ─────────────
X_real_norm = (X_real - X_mean) / X_std
X_tensor    = torch.tensor(X_real_norm)

# ── Run inference ────────────────────────────────────────────────────
with torch.no_grad():
    probs = model(X_tensor).numpy().flatten()
    preds = (probs > 0.5).astype(int)

# ── Results ──────────────────────────────────────────────────────────
accuracy = (preds == y_real).mean()

TP = ((preds == 1) & (y_real == 1)).sum()
TN = ((preds == 0) & (y_real == 0)).sum()
FP = ((preds == 1) & (y_real == 0)).sum()
FN = ((preds == 0) & (y_real == 1)).sum()

print(f"\n{'='*35}")
print(f"  Patients tested : {len(y_real)}")
print(f"  Accuracy        : {accuracy*100:.1f}%")
print(f"{'='*35}")
print(f"  True Positives  : {TP}  (diabetic, correctly caught)")
print(f"  True Negatives  : {TN}  (healthy, correctly cleared)")
print(f"  False Positives : {FP}  (healthy, wrongly flagged)")
print(f"  False Negatives : {FN}  (diabetic, missed)")
print(f"{'='*35}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded!


Saving diabetes.csv to diabetes (2).csv

  Patients tested : 768
  Accuracy        : 50.1%
  True Positives  : 265  (diabetic, correctly caught)
  True Negatives  : 120  (healthy, correctly cleared)
  False Positives : 380  (healthy, wrongly flagged)
  False Negatives : 3  (diabetic, missed)
